# R-tag pipeline — single run walkthrough

This notebook runs the **Python R-tag (RTEG) workflow** end-to-end for the KB331 sample filter, showcasing what each script and input file does.

**Ground truth:** manual flow in Virtuoso via `rdsBawTEGAutoFromTemp.il`  
**Docs:** [`workflow.md`](workflow.md) · [`README.md`](README.md) · [`../CLAUDE.md`](../CLAUDE.md)

| Phase | Scripts | Purpose |
|---|---|---|
| **Inputs** | GDS + layermap + JSON | Filter, frame, probe pad, golden reference |
| **Foundation** | `layermap.py`, `inspect_refs.py`, `separate.py`, `rteg_skill.py`, `build_rteg.py` | Read GDS, identify resonators, visual export |
| **Prepare** | `prepare_rteg.py`, `inspect_golden.py`, `geometry.py` | Template assembly — frame at top-left, signal node at center (all 8) |
| **Route (v1)** | `route_rteg.py` | Signal route + ground recut + DRC (S3 demo) |

Run all cells top-to-bottom from the `python_code/` directory (or adjust `ROOT` below).

In [52]:
from __future__ import annotations

import io
import json
import sys
import warnings
from contextlib import redirect_stdout
from pathlib import Path

import gdstk
import pandas as pd

import importlib

# Notebook lives in python_code/ — ensure imports resolve
ROOT = Path.cwd() if (Path.cwd() / "layermap").exists() else Path.cwd() / "python_code"
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))


def reload_project_modules() -> None:
    """Reload local .py modules so kernel picks up file edits without restart."""
    import build_rteg
    import inspect_golden
    import prepare_rteg
    import route_rteg
    import rteg_skill
    import separate

    for mod in (separate, rteg_skill, build_rteg, prepare_rteg, inspect_golden, route_rteg):
        importlib.reload(mod)


reload_project_modules()

DRAFT = ROOT / "draft_output"
GOLDEN = ROOT / "example_output" / "KB331_N_01_RTEG1_S3.gds"
FILTER = ROOT / "KB331_N_01_clean.gds"
FRAME = ROOT / "KB331_N_Frame.gds"
PPD = ROOT / "ppd_1port.gds"
INST_MAP = ROOT / "resonator_inst_map.json"

print(f"Working directory: {ROOT}")
print(f"Draft output:      {DRAFT}")

Working directory: c:\Users\santosr4\Documents\rTEG Automation\python_code
Draft output:      c:\Users\santosr4\Documents\rTEG Automation\python_code\draft_output


## 1. Input files

These are the files every later step depends on. Nothing in `example_output/` is written by Python.

In [53]:
INPUTS = [
    ("Filter layout", FILTER, "Clean hierarchical filter GDS — resonators + connect metal"),
    ("Frame template", FRAME, "460×580 µm GSG probe frame (six BAW_MB2 pads)"),
    ("Probe device", PPD, "ppd_1port — SKILL single-port pad reference"),
    ("Layer table", ROOT / "layermap", "Skyworks name → GDS (layer, datatype)"),
    ("Golden S3", GOLDEN, "Read-only reference RTEG for resonator S3"),
    ("Inst-name overrides", INST_MAP, "Optional index → Virtuoso instName (e.g. 6→S3)"),
]

rows = []
for label, path, role in INPUTS:
    exists = path.exists()
    size = f"{path.stat().st_size:,} B" if exists and path.is_file() else "—"
    rows.append({"file": label, "path": path.name, "exists": exists, "size": size, "role": role})

pd.DataFrame(rows)

,file,path,exists,size,role
0,Filter layout,KB331_N_01_clean.gds,True,"655,360 B",Clean hierarchical filter GDS — resonators + c...
1,Frame template,KB331_N_Frame.gds,True,"34,816 B",460×580 µm GSG probe frame (six BAW_MB2 pads)
2,Probe device,ppd_1port.gds,True,"8,192 B",ppd_1port — SKILL single-port pad reference
3,Layer table,layermap,True,"3,971 B","Skyworks name → GDS (layer, datatype)"
4,Golden S3,KB331_N_01_RTEG1_S3.gds,True,"102,400 B",Read-only reference RTEG for resonator S3
5,Inst-name overrides,resonator_inst_map.json,True,48 B,Optional index → Virtuoso instName (e.g. 6→S3)


## 2. `layermap.py` — layer name lookups

All geometry scripts use **layer names** (`BAW_MBE`, `BAW_MTE`, …) instead of hardcoded GDS numbers.

In [54]:
from layermap import LAYERMAP_PATH, load_layermap, describe_layers

layermap = load_layermap()
print(f"Loaded {len(layermap)} layers from {LAYERMAP_PATH.name}")

for name in ("BAW_MBE", "BAW_MTE", "BAW_MB2", "BAW_EDGE"):
    layer, dt = layermap.pair(name)
    print(f"  {name:12s} → GDS ({layer}, {dt})")

Loaded 155 layers from layermap
  BAW_MBE      → GDS (2, 0)
  BAW_MTE      → GDS (5, 0)
  BAW_MB2      → GDS (12, 0)
  BAW_EDGE     → GDS (9, 0)


## 3. `inspect_refs.py` — hierarchy sanity check

Lists placed references in the filter GDS: resonators, vias, connect cells. Useful when instance names did not survive export.

In [55]:
from inspect_refs import inspect_gds

buf = io.StringIO()
with redirect_stdout(buf):
    inspect_gds(FILTER)
print(buf.getvalue()[:2000])
if len(buf.getvalue()) > 2000:
    print("… (truncated)")


=== KB331_N_01_clean.gds ===

shuntq3_CDNS_780903262810: 80 polys, bbox (-127.4, -49.5)-(128.1, 54.5) (no references)
   labels (9):
     'P1' @ (0.0, -0.0)  layer=(5,0)
     'P2' @ (0.0, -0.0)  layer=(2,0)
     'freq=1478M INFRAver=3.0 ModelID=430 Band=nil multiKt2=None pcType=Q' @ (0.0, 0.0)  layer=(100,0)
     'ORaW=3.4' @ (0.0, 9.5)  layer=(221,0)
     '[@instanceName]' @ (0.0, 36.0)  layer=(221,0)
     'ReW=-99' @ (0.0, 1.5)  layer=(221,0)
     'MRaW=2.2' @ (0.0, -6.5)  layer=(221,0)
     'MTE_CLEN=172.6' @ (0.0, -14.5)  layer=(221,0)
     'MBE_CLEN=377.9' @ (0.0, -22.5)  layer=(221,0)

seriesq3_CDNS_780903262811: 93 polys, bbox (-37.6, -53.2)-(43.3, 53.2) (no references)
   labels (9):
     'P1' @ (0.0, 0.0)  layer=(5,0)
     'P2' @ (0.0, 0.0)  layer=(2,0)
     'freq=1541M INFRAver=3.0 ModelID=430 Band=nil multiKt2=None pcType=Q' @ (0.0, 0.0)  layer=(100,0)
     'ORaW=3.4' @ (0.0, 5.0)  layer=(221,0)
     '[@instanceName]' @ (0.0, 10.0)  layer=(221,0)
     'ReW=1.8' @ (0.0, 0.0)

## 4. `separate.py` — resonator identification

Ports SKILL lines 178–179: find instances whose master starts with `series`, `shunt`, `rcap`, or `mimcap`. Also finds `vtb` vias.

In [56]:
from separate import separate, find_vias, group_splits

filter_lib = gdstk.read_gds(FILTER)
by_parent = separate(filter_lib)
parent, res_list = next(iter(sorted(by_parent.items())))
filter_cell = next(c for c in filter_lib.cells if c.name == parent)
vias = find_vias(filter_cell)
groups = group_splits(res_list)

print(f"Parent cell: {parent}")
print(f"Resonators: {len(res_list)}  |  Groups: {len(groups)}  |  Vias: {len(vias)}")

df_res = pd.DataFrame([
    {
        "index": i,
        "inst_name": r.inst_name[:28],
        "type": r.res_type,
        "origin_x": round(r.origin[0], 1),
        "origin_y": round(r.origin[1], 1),
        "rotation_deg": round(r.rotation * 180 / 3.141592653589793, 1),
    }
    for i, r in enumerate(res_list)
])
df_res

Parent cell: KB331_N_01
Resonators: 8  |  Groups: 6  |  Vias: 4


,index,inst_name,type,origin_x,origin_y,rotation_deg
0,0,shuntq3_CDNS_780903262810,shunt,282.6,183.1,0.0
1,1,shuntq3_CDNS_780903262810,shunt,234.0,98.3,180.0
2,2,seriesq3_CDNS_780903262811,series,95.8,145.1,90.0
3,3,seriesq3_CDNS_780903262811,series,92.0,217.4,270.0
4,4,shuntq3_CDNS_780903262812,shunt,311.9,458.9,90.0
5,5,shuntq3_CDNS_780903262815,shunt,157.8,412.7,0.0
6,6,seriesq3_CDNS_780903262817,series,307.6,317.6,270.0
7,7,seriesq3_CDNS_780903262818,series,187.7,294.1,270.0


## 5. `rteg_skill.py` — shared preprocessing helpers

Frame and ppd anchor at the **top-left (0, 0)** of the RTEG cell. The resonator is shifted so its **signal node** (BAW_MTE centroid for series, BAW_MBE for shunt) lands on the frame center.

- **`placement_shift()`** — signal feed centroid → frame bbox center
- **Inst-name inference** (sorted placement → S1/P1/…) + JSON overrides
- **`connect_backup`** loader with MTE/MBE fallback

In [57]:
reload_project_modules()

from layermap import load_layermap
from rteg_skill import (
    _bbox_center,
    _world_layer_centroid,
    frame_top_cell,
    infer_inst_names,
    load_connect_backup,
    placement_shift,
    rteg_cell_name,
)

layermap = load_layermap()
frame_lib = gdstk.read_gds(FRAME)
frame_cell = frame_top_cell(frame_lib)
sample_res = res_list[6]
dx, dy = placement_shift(sample_res, frame_cell, layermap)

inst_names = infer_inst_names(res_list, INST_MAP)
backup = load_connect_backup(parent, ROOT, filter_lib)

frame_bb = frame_cell.bounding_box()
fcx, fcy = _bbox_center(frame_bb)
signal_xy = _world_layer_centroid(sample_res, "BAW_MTE", layermap)
shifted_signal = (signal_xy[0] + dx, signal_xy[1] + dy) if signal_xy else None

print(f"Frame bbox: {frame_bb}  center=({fcx:.1f}, {fcy:.1f})")
print(f"Sample placement shift (dx, dy) for index 6: ({dx:.1f}, {dy:.1f})")
if shifted_signal:
    print(f"Signal MTE after shift: ({shifted_signal[0]:.1f}, {shifted_signal[1]:.1f})")
print(f"connect_backup loaded: {backup is not None}")
print(f"Overrides file: {json.loads(INST_MAP.read_text())}")

pd.DataFrame([
    {"index": i, "inferred_inst": inst_names[i], "cell_name": rteg_cell_name(parent, inst_names[i])}
    for i in range(len(res_list))
])

Frame bbox: ((0.0, 0.0), (460.0, 580.0))  center=(230.0, 290.0)
Sample placement shift (dx, dy) for index 6: (-63.7, -18.0)
Signal MTE after shift: (230.0, 290.0)
connect_backup loaded: False
Overrides file: {'overrides_by_index': {'6': 'S3'}}


,index,inferred_inst,cell_name
0,0,P3,KB331_N_01_RTEG1_P3
1,1,P4,KB331_N_01_RTEG1_P4
2,2,S4,KB331_N_01_RTEG1_S4
3,3,S5,KB331_N_01_RTEG1_S5
4,4,P1,KB331_N_01_RTEG1_P1
5,5,P2,KB331_N_01_RTEG1_P2
6,6,S3,KB331_N_01_RTEG1_S3
7,7,S2,KB331_N_01_RTEG1_S2


## 6. `build_rteg.py` — visual export (no preserved metal)

One GDS per resonator: frame + ppd at top-left, resonator shifted so the signal node sits at frame center. Quick KLayout check that separation picked the right geometry.

In [58]:
reload_project_modules()
from build_rteg import export_resonators

build_dir = DRAFT / "notebook_build"
build_stats = export_resonators(FILTER, build_dir)

pd.DataFrame([
    {
        "cell": s.cell_name,
        "inst": s.inst_name,
        "type": s.res_type,
        "filter@": s.filter_origin,
        "rteg@": tuple(round(x, 1) for x in s.rteg_origin),
    }
    for s in build_stats
])

,cell,inst,type,filter@,rteg@
0,KB331_N_01_RTEG1_P3,P3,shunt,"(282.6, 183.1)","(279.2, 285.6)"
1,KB331_N_01_RTEG1_P4,P4,shunt,"(234.0, 98.3)","(180.8, 294.4)"
2,KB331_N_01_RTEG1_S4,S4,series,"(95.8, 145.1)","(210.8, 287.2)"
3,KB331_N_01_RTEG1_S5,S5,series,"(92.0, 217.4)","(249.2, 292.8)"
4,KB331_N_01_RTEG1_P1,P1,shunt,"(311.90000000000003, 458.90000000000003)","(201.9, 322.6)"
5,KB331_N_01_RTEG1_P2,P2,shunt,"(157.8, 412.7)","(253.3, 259.6)"
6,KB331_N_01_RTEG1_S3,S3,series,"(307.6, 317.6)","(243.9, 299.6)"
7,KB331_N_01_RTEG1_S2,S2,series,"(187.70000000000002, 294.1)","(211.3, 282.1)"


## 7. `prepare_rteg.py` — routing input assembly

Builds the full pre-route cell:
1. Frame + `ppd_1port` at top-left `(0, 0)`
2. Resonator shifted so **signal node** lands on frame center (BAW_MTE for series, BAW_MBE for shunt)
3. Preserved connect metal + nearby vias (same shift as resonator)
4. Trim to golden layer set (drops BF/TSV/EM_VPT fill)

Below: prepare **golden anchor S3** (filter index 6).

In [59]:
reload_project_modules()
from prepare_rteg import prepare_resonator, prepare_all

S3_INDEX = 6

with warnings.catch_warnings(record=True) as w:
    warnings.simplefilter("always")
    prepared_path, prep_stats = prepare_resonator(S3_INDEX, golden_gds=GOLDEN, output_dir=DRAFT)

print(f"Output: {prepared_path.name}")
print(f"  inst_name={prep_stats.inst_name}")
print(f"  filter@={tuple(round(x,1) for x in prep_stats.filter_origin)}")
print(f"  rteg@={tuple(round(x,1) for x in prep_stats.rteg_origin)}  (signal node -> frame center)")
print(f"  metal: {prep_stats.preserved_poly_count} polys ({prep_stats.metal_source})")
print(f"  vias={prep_stats.via_count}  trimmed={prep_stats.trimmed_poly_count} polys")
print(f"  layers absent from golden: {prep_stats.layers_absent_from_golden or 'none'}")
for msg in w:
    print(f"  WARNING: {msg.message}")

# Verify top-cell structure
prep_lib = gdstk.read_gds(prepared_path)
top = next(c for c in prep_lib.cells if c.name == prep_stats.cell_name)
print(f"\nTop cell bbox: {top.bounding_box()}")
print(f"References ({len(top.references)}):")
for ref in top.references:
    name = ref.cell.name if ref.cell else "?"
    print(f"  {name[:36]:36s} @ {tuple(round(x,1) for x in ref.origin)}")

Output: KB331_N_01_RTEG1_S3_prepared.gds
  inst_name=S3
  filter@=(307.6, 317.6)
  rteg@=(243.9, 299.6)  (signal node -> frame center)
  metal: 44 polys (connectMTE/MBE)
  vias=0  trimmed=100 polys
  layers absent from golden: none

Top cell bbox: ((0.0, 0.0), (460.0, 580.0))
References (3):
  KB331_N_Frame                        @ (0.0, 0.0)
  ppd_1port                            @ (0.0, 0.0)
  seriesq3_CDNS_780903262817           @ (243.9, 299.6)


In [60]:
# Optional: batch all 8 resonators (same as `python prepare_rteg.py --all`)
RUN_BATCH = False  # set True to regenerate all prepared files

if RUN_BATCH:
    with warnings.catch_warnings(record=True) as w:
        warnings.simplefilter("always")
        batch = prepare_all(golden_gds=GOLDEN, output_dir=DRAFT)
    pd.DataFrame([
        {"file": p.name, "inst": s.inst_name, "rteg@": tuple(round(x,1) for x in s.rteg_origin)}
        for p, s in batch
    ])
else:
    print("Set RUN_BATCH=True to run prepare_all() for all 8 resonators.")

Set RUN_BATCH=True to run prepare_all() for all 8 resonators.


## 8. `inspect_golden.py` — prepared vs golden diff

Read-only diagnostic: layer inventory, MBE/MTE extents, and what the golden has that prepare does not yet generate (fill/trim layers).

In [61]:
from inspect_golden import build_notes, grouped_missing_layers

notes = build_notes(GOLDEN, prepared_path)
print(notes)

missing = grouped_missing_layers(GOLDEN, prepared_path)
pd.DataFrame(missing) if missing else "No grouped missing layers."

## NOTES — golden vs prepared

- golden:   `KB331_N_01_RTEG1_S3.gds` — 163 polys, 83 layer pairs
- prepared: `KB331_N_01_RTEG1_S3_prepared.gds` — 316 polys, 62 layer pairs
- golden bbox: (0.0, 0.0) - (460.0, 580.0)
- prepared bbox: (0.0, 0.0) - (460.0, 580.0)

### Metal extents (flattened)
- BAW_MBE: golden 3 polys (9.5, 9.5) - (450.5, 570.5) | prepared 19 polys (7.0, 7.0) - (450.5, 570.5)
- BAW_MTE: golden 1 polys (139.5, 232.3) - (288.0, 349.2) | prepared 4 polys (59.1, 234.9) - (289.1, 366.3)

### Layers in golden, absent from prepared (21)
These are routing/ground/fill layers the manual flow adds; v1 routes only MBE/MTE.

| layer | gds | polys |
|---|---|---|
| BAW_MF3 | 42/0 | 1 |
| BAW_MF4 | 43/0 | 1 |
| BAW_MF5 | 44/0 | 1 |
| BAW_MF6 | 45/0 | 1 |
| BAW_MF7 | 46/0 | 1 |
| BAW_MF8 | 47/0 | 1 |
| BAW_H4 | 201/0 | 1 |
| BAW_H6 | 203/0 | 13 |
| BAW_H7 | 204/0 | 3 |
| BAW_H17 | 217/0 | 3 |
| BAW_H21 | 221/0 | 3 |
| BAW_H22 | 222/0 | 3 |
| BAW_NoTF1 | 227/0 | 1 |
| BAW_NoTF4 | 230/0 | 

,group,layers,poly_count,tag
0,H-series (hole/fill),"BAW_H17, BAW_H21, BAW_H22, BAW_H4, BAW_H6, BAW_H7",26,likely derived (fill/trim/hole)
1,No-series (mask exclusions),"BAW_NoTF1, BAW_NoTF4, BAW_NoTF5, BAW_NoTF6, BA...",7,likely derived
2,MF-series (metal fill),"BAW_MF3, BAW_MF4, BAW_MF5, BAW_MF6, BAW_MF7, B...",6,likely copied from source / derived
3,TEG/ORF/label,"BAW_ORF, BAW_TEG",3,unknown -- SME


## 9. `geometry.py` — shared boolean / net helpers

Not run directly. Used by `prepare_rteg.py` (layer trim) and `route_rteg.py` (routable region, connectors, DRC nets).

In [62]:
import geometry as G

prep_top = gdstk.read_gds(prepared_path).cells[-1]
flat = G.flatten_cell(prep_top)
print(f"Flattened prepared top: {len(flat)} polygons")

# Layer trim helper (same allow-list as prepare)
from route_rteg import resolve_allowed_layer_pairs, RouteConfig
allowed = resolve_allowed_layer_pairs(RouteConfig(), GOLDEN)
print(f"Golden allow-list: {len(allowed)} layer pairs")

Flattened prepared top: 316 polygons
Golden allow-list: 83 layer pairs


## 10. `route_rteg.py` — routing v1 (S3 demo)

Draws a simple signal connector, recuts frame ground, runs net-aware DRC, writes routed GDS + net overlay + report.

Set `RUN_ROUTE = True` to execute (takes a few seconds). Uses the SKILL-named prepared file from step 7.

In [63]:
RUN_ROUTE = True  # set False to skip routing

if RUN_ROUTE:
    reload_project_modules()
    from route_rteg import route, write_report, RouteConfig

    prepared_path = Path(prepared_path) if "prepared_path" in globals() else DRAFT / "KB331_N_01_RTEG1_S3_prepared.gds"
    if not prepared_path.is_file():
        raise FileNotFoundError(f"Missing {prepared_path} — run the prepare cell (section 7) first.")

    cfg = RouteConfig()
    result = route(prepared_path, GOLDEN, cfg)
    report_path = write_report(result, cfg, prepared_path, GOLDEN)

    print("=== Route result ===")
    print(f"Prepared:    {prepared_path.name}")
    print(f"Routed GDS:  {result.routed_path}")
    print(f"Net overlay: {result.nets_path}")
    print(f"Report:      {report_path}")
    print(f"Signal route drawn: {result.signal_routed} ({result.signal_kind or 'n/a'})")
    print(f"DRC introduced violations: {result.real_drc_violations}")
    for layer in ("BAW_MBE", "BAW_MTE"):
        pct = result.overlap.get(layer, {}).get("overlap_fraction_of_golden", 0) * 100
        print(f"{layer} overlap vs golden: {pct:.1f}%")
else:
    print("Routing skipped (RUN_ROUTE=False).")

=== Route result ===
Prepared:    KB331_N_01_RTEG1_S3_prepared.gds
Routed GDS:  c:\Users\santosr4\Documents\rTEG Automation\python_code\draft_output\KB331_N_01_RTEG1_S3_routed.gds
Net overlay: c:\Users\santosr4\Documents\rTEG Automation\python_code\draft_output\KB331_N_01_RTEG1_S3_nets.gds
Report:      c:\Users\santosr4\Documents\rTEG Automation\python_code\draft_output\ROUTE_S3_REPORT.md
Signal route drawn: True (straight)
DRC introduced violations: 0
BAW_MBE overlap vs golden: 41.3%
BAW_MTE overlap vs golden: 65.5%


## 11. Output files in `draft_output/`

Open GDS in **KLayout**. Use `*_nets.gds` with `*_nets.lyp` to visualize net classification from routing.

In [64]:
DRAFT.mkdir(parents=True, exist_ok=True)

outputs = sorted(DRAFT.glob("*"), key=lambda p: p.stat().st_mtime, reverse=True)
rows = []
for p in outputs[:25]:
    kind = "GDS" if p.suffix == ".gds" else p.suffix.lstrip(".") or "file"
    rows.append({"name": p.name, "kind": kind, "size_KB": round(p.stat().st_size / 1024, 1)})

pd.DataFrame(rows) if rows else "No files in draft_output yet."

,name,kind,size_KB
0,ROUTE_S3_REPORT.md,md,12.7
1,KB331_N_01_RTEG1_S3_nets.lyp,lyp,1.5
2,KB331_N_01_RTEG1_S3_nets.gds,GDS,161.1
3,KB331_N_01_RTEG1_S3_routed.gds,GDS,157.3
4,KB331_N_01_RTEG1_S3_prepared.gds,GDS,156.3
5,notebook_build,file,4.0


## Quick reference — CLI equivalents

```powershell
cd python_code
python layermap.py
python inspect_refs.py
python separate.py
python build_rteg.py
python prepare_rteg.py --all
python inspect_golden.py --prepared draft_output/KB331_N_01_RTEG1_S3_prepared.gds
python route_rteg.py --prepared draft_output/KB331_N_01_RTEG1_S3_prepared.gds
```

**Open in KLayout:**
- `draft_output/KB331_N_01_RTEG1_S3_prepared.gds` — pre-route SKILL assembly
- `draft_output/KB331_N_01_RTEG1_*_routed.gds` — routed draft (if step 10 ran)
- `example_output/KB331_N_01_RTEG1_S3.gds` — golden reference